# 01 — Dataset Overview

This notebook gives an interactive overview of the CFB-GBM dataset after downloading metadata.
Run `download_tcia.py --mode metadata` and `inspect_dicom.py` first.

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from utils import load_config

cfg = load_config('../config/config.yaml')
sns.set_style('whitegrid')
%matplotlib inline

In [ ]:
meta_path = Path('../data/processed/series_metadata.csv')
if not meta_path.exists():
    print('Run: python src/inspect_dicom.py first')
else:
    df = pd.read_csv(meta_path)
    print(f'Total series: {len(df)}')
    print(f'Total patients: {df["PatientID"].nunique()}')
    display(df.head())

In [ ]:
# Modality distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['Modality'].value_counts().plot.bar(ax=axes[0], color='steelblue', title='Series by Modality')
mr = df[df['Modality'] == 'MR']
mr['SequenceType'].value_counts().plot.bar(ax=axes[1], color='salmon', title='MR Sequence Types')
for ax in axes:
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('../results/figures/dataset_modality_dist.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Per-patient completeness matrix
sequences = ['T1Gd', 'FLAIR', 'T1', 'T2', 'DWI']
patients = df['PatientID'].unique()
completeness = pd.DataFrame(index=patients, columns=sequences, data=False)
for seq in sequences:
    present = df[df['SequenceType'] == seq]['PatientID'].unique()
    completeness.loc[present, seq] = True

fig, ax = plt.subplots(figsize=(8, max(4, len(patients) * 0.2)))
sns.heatmap(completeness.astype(int), cmap='YlGn', cbar=False,
            linewidths=0.5, linecolor='gray', ax=ax)
ax.set_title('Per-patient MRI Sequence Availability')
ax.set_xlabel('Sequence')
ax.set_ylabel('Patient')
plt.tight_layout()
plt.savefig('../results/figures/completeness_matrix.png', dpi=120, bbox_inches='tight')
plt.show()

print('\nCompleteness summary:')
print(completeness.sum())

In [ ]:
# Slice count distribution for T1Gd
t1gd = df[df['SequenceType'] == 'T1Gd']
fig, ax = plt.subplots(figsize=(7, 4))
t1gd['NumSlices'].hist(bins=20, ax=ax, color='#4e79a7', edgecolor='white')
ax.set_xlabel('Number of Slices')
ax.set_ylabel('Count')
ax.set_title('T1Gd Series: Slice Count Distribution')
plt.tight_layout()
plt.show()